# SSMR-Bench Train_visual → Qwen3-VL-4B GRPO (Colab A100 40GB)

VS Code에서 열고 커널을 **Colab 런타임 → A100**으로 선택합니다.
Colab 유닛(100개 ≈ A100 8.5시간)은 **이 노트북의 학습에만** 씁니다. 평가는 실습실 RTX 4090에서 합니다.
순서: 1~4 준비 → 5 파일럿(10스텝) → 6 본 학습 → 7 merge → **7 직후 런타임 해제**.

## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다"
name = torch.cuda.get_device_name(0); vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(name, f"{vram:.1f} GB")

## 2. 코드 받기 (GitHub `ssmr` 브랜치)
저장소가 public이라 토큰 없이 clone됩니다. 코드를 고친 뒤에는 이 셀만 다시 실행하면 최신으로 받습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO    = 'https://github.com/Groo-2/MMDL.git'
BRANCH  = 'ssmr'
PROJECT = '/content/drive/MyDrive/mmmu_project'          # 체크포인트와 결과
RUN     = f'{PROJECT}/grpo_run1'
WORK    = '/content/MMDL'
DATA    = '/content/ssmr'                                  # 변환된 데이터 (로컬 디스크)

!mkdir -p {PROJECT} {RUN}
!rm -rf {WORK} && git clone -q -b {BRANCH} --depth 1 {REPO} {WORK}
!ls {WORK} {WORK}/music_ft

## 3. 설치 (끝나면 커널 재시작 후 4번부터)

In [ ]:
!pip install -q -r /content/MMDL/music_ft/requirements_train.txt
# Colab 기본 torchao(0.10.0)가 PEFT 요구 버전(>=0.16)보다 낮아 LoRA 생성에서 ImportError가 난다. 우리는 쓰지 않으므로 제거
!pip uninstall -y -q torchao
!pip freeze > /content/drive/MyDrive/mmmu_project/requirements_train.lock.txt

## 4. 데이터 변환 + 중복 제거 (CPU만 사용, 약 5분)

In [ ]:
# 3번 셀 뒤에 커널을 재시작하면 변수가 사라지므로, 이후 셀이 쓰는 경로를 여기서 모두 다시 정의한다
PROJECT = '/content/drive/MyDrive/mmmu_project'
RUN     = f'{PROJECT}/grpo_run1'
WORK    = '/content/MMDL'
DATA    = '/content/ssmr'
!python {WORK}/music_ft/build_ssmr.py --out-dir {DATA} --cache-dir /content/hf_cache
!python {WORK}/music_ft/dedupe.py --data-dir {DATA} --cache-dir /content/hf_cache
!cp {DATA}/dedupe_report.json /content/drive/MyDrive/mmmu_project/ssmr_dedupe_report.json
!cd {WORK}/music_ft && python -m unittest test_reward

## 4-1. cuDNN 점검 (1분)
비전 패치 임베딩(conv3d)이 이 환경에서 도는지 확인합니다. 실패하면 어떤 설정에서 통과하는지 요약에 나옵니다.


In [ ]:
!python {WORK}/music_ft/diag_cudnn.py


## 5. 파일럿 (3스텝) — 스텝당 시간, 메모리, 보상, 응답 길이 확인
여기서 **TRL + Qwen3-VL + 이미지 여러 장 + vLLM colocate가 실제로 도는지**를 확인합니다.
로그에서 볼 것: `step 시간`, `reward`, `completions/mean_length`, `completions/clipped_ratio`, GPU 메모리.
기본 micro-batch는 2입니다(이미지 5장짜리 문항에서 메모리 부족을 피하려고). 여유가 있으면 4로 올려 속도를 비교하세요.

In [ ]:
!cd {WORK}/music_ft && python train_grpo.py --data-dir {DATA} --output-dir {RUN}/pilot \
    --max-steps 3 --prompts-per-step 16 --save-steps 3

## 6. 본 학습
파일럿 결과로 스텝 수를 정합니다(목표: Colab 약 3.5~4시간 ≈ 45유닛). 세션이 끊기면 같은 셀을 `--resume`으로 다시 실행합니다.

In [ ]:
MAX_STEPS = 60   # 파일럿의 스텝당 시간을 보고 수정
!cd {WORK}/music_ft && python train_grpo.py --data-dir {DATA} --output-dir {RUN} \
    --max-steps {MAX_STEPS} --prompts-per-step 16 --save-steps 10 --resume

## 7. merge (bf16) → Drive에 저장. 끝나면 **런타임 연결 해제** (유닛 절약)

In [ ]:
!cd {WORK}/music_ft && python merge_lora.py --lora {RUN}/final_lora --out {RUN}/merged